 # 📍 **MovieLens Recommendation System**

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

In [3]:
ratings = pd.read_csv('./data/ratings.csv') 
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
movies = pd.read_csv('./data/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tags = pd.read_csv('./data/tags.csv')
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
links = pd.read_csv('./data/links.csv')
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


**Problem**: Predict how a user would rate an unseen movie. (i.e.) our target variable is rating. For the data part. *ratings.csv* is our core. *movies.csv* has genre details and *tags.csv* might also be useful. I don't we will ever need *links.csv*

# **Data Inspection**

In [7]:
#What are the dimensions of the dataset? (number of rows and columns)

ratings.shape

(100836, 4)

ratings dataframe have *1,00,836 rows* and *4 columns*

In [8]:
#What are the data types of each column and are there any missing values?

ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


The *timestamp* dtype is in *int64* which is weird. - (We will look into it later)

In [9]:
#What are the summary statistics of the dataset? (mean, median, standard deviation, etc.)

ratings.describe()

,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


Initial guess my looking here we can say that there might be *610 users* and *1,93,609 movies* - Important Trap : That would be true only if we make sure it is sequential. Due to that fact that there are only *1,00,836 rows* then having *1,93,609 movies* is not possible. But, we can't conclude anything about userId. 

In [10]:
# Checking if the userId and movieId columns are sequantial or if there are any missing values in the sequence

print(f" Unique movies: {ratings['movieId'].nunique()} and Unique users: {ratings['userId'].nunique()}")

 Unique movies: 9724 and Unique users: 610


This confirms that we have *610 Users* and *9724 Movies*

**Target Variable Stats** : The rating columns ranges from *0.5 - 5.0* and it has **mean of 3.5** (i.e.) rating above 3.5 should be modeled as good and bad if otherwise.

In [11]:
# What is the shape of the movies dataset?

movies.shape

(9742, 3)

movies dataframe has *9742* rows and *3* columns.

In [12]:
# What are the data types of each column and are there any missing values?

movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB


No null values in movies dataframe and the data types looks fine too.

In [13]:
# Checking if movieId column is a unique identifier in this dataset

movies['movieId'].nunique()

9742

Hence, That proves that this dataframe had *movieId* as its unique identifier. Because the number of rows and number of unique values in *movieId* are equal.

**Important Note** : The ratings df has *9,724* unique values for *movieId* and movies df has *9,742* unique values for *movieId* and therefore, we have 18 non-reviewed movies. 

In [14]:
# What is the shape of the tags dataset?

tags.shape

(3683, 4)

tags dataframe has *3683* rows and *4* columns

In [15]:
# What are the data types of each column and are there any missing values?

tags.info()

<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 115.2 KB


tags dataframe has no null values and also other than timestamp which we already noticed. There is no weird behaviour here and also based on the row count. I can't see there isn't a unique indentifier.

**Important Note** : Here there are only *3,683* rows and in ratings df had *1,00,836* rows. Means we only have tags for *3.6%* rows in the core ratings df. So, There seems to be a problem here.

In [20]:
# Storing the count description into a dictionary, so that it can be used later

data = {
    "ratings" : {
        "rows" : ratings.shape[0],
        "columns" : ratings.shape[1],
        "userId" : ratings['userId'].nunique(),
        "movieId" : ratings['movieId'].nunique()
    },
    "movies" : {
        "rows" : movies.shape[0],
        "columns" : movies.shape[1],
        "movieId" : movies['movieId'].nunique()
    },
    "tags" : {
        "rows" : tags.shape[0],
        "columns" : tags.shape[1],
        "userId" : tags['userId'].nunique(),
        "movieId" : tags['movieId'].nunique()
    }
}

# **Data Audit**

In [ ]:
# Audit on userId and moveId value counts to check if there are integrity issues

print(f"Number of unique users in ratings: {data['ratings']['userId']}, tags: {data['tags']['userId']}")
print(f"Number of unique movies in ratings: {data['ratings']['movieId']}, tags: {data['tags']['movieId']}")
print(f"Number of unique movies in movies: {data['movies']['movieId']}")

Number of unique users in ratings: 610, tags: 58
Number of unique movies in ratings: 9724, tags: 1572
Number of unique movies in movies: 9742


*movieId* is a unique indentifier in movies df with *9,742* values and there are only *9,724* movies in ratnings. That means there are 18 movies that hasn't been reviewed yet.

**Problem** : Here if we merge the ratings and movies df. Then we will have null values and that too for the *Target Variable* itself. It's called a **Cold Start** problem. Because, you don't have data for the model to train upon.

In [ ]:
# Looking for duplicated rows across datasets, because duplicates won't be useful

print(f"Number of duplicated rows in ratings: {ratings.duplicated().sum()}")
print(f"Number of duplicated rows in movies: {movies.duplicated().sum()}")
print(f"Number of duplicated rows in tags: {tags.duplicated().sum()}")

Number of duplicated rows in ratings: 0
Number of duplicated rows in movies: 0
Number of duplicated rows in tags: 0


No fully duplicated rows among our datasets. But, there can be multiple reviews for the same movie. 

In [ ]:
# Checking if there are multiple reviews from same user to the same movie. (i.e.) User reviewing movie multiple times
# This would help us in while spliting the dataset more meaningfully in the future.

ratings[['userId', 'movieId']].value_counts().max()

np.int64(1)

No multiple reviews from the same user to same movie.

In [ ]:
# Checking if there are movies reviewed but not in movies dataset. 
# If it is then it would be integrity issue that would later cause us missing values.

ratings['movieId'].isin(movies['movieId']).sum() == ratings.shape[0]

np.True_

No movie reviewed without being in movies df.

In [ ]:
# Auditing the target variable if it's all valid. 
# Only valid values are 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5

ratings['rating'].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

So, *rating* has no invalid values